In [65]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.graph import StateGraph, MessagesState, START, END
from typing import TypedDict,Annotated,Literal
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver
from uuid import uuid4

load_dotenv()


True

In [66]:
llm=ChatGroq(model='openai/gpt-oss-120b')

In [67]:
class PostState(TypedDict):
    title: str
    body: str
    decision: str

In [68]:
def generate_post(state: PostState) -> dict[str, str]:
    prompt="Generate a porffesional  yet appealing  on gaming in india based on x.com recent posts with only title"
    title = llm.invoke(prompt).content
    return {"title": title}
    
def generate_body(state: PostState) -> dict[str, str]:
    title = state['title']
    prompt=f"Generate a porffesional  yet appealing  on title {title} in india based on x.com recent posts with only body "
    body = llm.invoke(prompt).content
    return {"body": body}

def accept_or_reject(state: PostState) -> dict[str, str]:
    decision = interrupt({
        "title": state["title"],
        "body": state["body"],
        "prompt": "Type approve or reject",
    })

    decision = str(decision).strip().lower()
    if decision not in {"approve", "reject"}:
        raise ValueError("Decision must be 'approve' or 'reject'.")

    return {"decision": decision}
    

In [69]:
graph = StateGraph(PostState)

graph.add_node("generate_post", generate_post)
graph.add_node("generate_body", generate_body)
graph.add_node("accept_or_reject", accept_or_reject)

graph.add_edge(START, "generate_post")
graph.add_edge("generate_post", "generate_body")
graph.add_edge("generate_body", "accept_or_reject")
graph.add_edge("accept_or_reject", END)

final = graph.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": str(uuid4())}}
result = final.invoke(
    {"title": "", "body": "", "decision": ""},
    config=config,
)

if "__interrupt__" not in result:
    raise RuntimeError("The graph did not pause for post review.")

review = result["__interrupt__"][0].value
print(f"\nTitle:\n{review['title']}\n\nBody:\n{review['body']}\n")

while True:
    decision = input("Type approve or reject, then press Enter: ").strip().lower()
    if decision in {"approve", "reject"}:
        break
    print("Please enter either 'approve' or 'reject'.")

result = final.invoke(Command(resume=decision), config=config)
print("Post approved." if result["decision"] == "approve" else "Post rejected.")


Title:
**"Leveling Up: How India's Gaming Boom Is Shaping the Global Market – Insights from the Latest X Trends"**

Body:
India’s gaming sector is no longer a niche hobby—it’s a powerhouse that’s redefining the global market. Recent chatter on X (formerly Twitter) reveals a confluence of factors—massive mobile adoption, robust venture capital inflows, and aggressive government support—that together are propelling the country from a consumer to a creator of worldwide gaming experiences. Below is a deep‑dive into the key trends emerging from the platform’s most recent posts, and what they mean for the industry at large.

---

### 1. Explosive User Growth – Numbers That Speak Volumes  

- **1.3 billion mobile connections**: A thread from @StatistaIndia (Oct 23) highlighted that India now boasts the world’s largest mobile user base, with over 70 % of the population online daily.  
- **350 million gamers**: According to a poll by @NikoPartner (Oct 22), the active gaming audience has surged